### Global Health Data Warehouse

This notebook implements a complete ETL pipeline for global health statistics including data extraction, transformation, quality validation, and warehouse deployment

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sqlite3
import requests
from datetime import datetime
import logging
import warnings
import json

warnings.filterwarnings('ignore')

### 1. ENVIRONMENT SETUP & CONFIGURATION

In [2]:
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('global_health_etl')

project_dir = Path.cwd()
data_dir = project_dir / "data"
warehouse_dir = project_dir / "data_warehouse"
logs_dir = project_dir / "logs"
exports_dir = warehouse_dir / "exports"

for directory in [data_dir, warehouse_dir, logs_dir, exports_dir]:
    directory.mkdir(exist_ok=True)

db_path = warehouse_dir / "global_health_datawarehouse.db"

pipeline_metadata = {
    'pipeline_name': 'Global Health Data Warehouse ETL',
    'version': '2.0',
    'execution_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'stages': {}
}

logger.info("=" * 70)
logger.info("Global Health Data Warehouse ETL Pipeline")
logger.info("=" * 70)
logger.info(f"Environment initialized successfully")
logger.info(f"Database location: {db_path}")

2025-11-14 22:16:37,454 - INFO - ======================================================================
2025-11-14 22:16:37,454 - INFO - Global Health Data Warehouse ETL Pipeline
2025-11-14 22:16:37,455 - INFO - ======================================================================
2025-11-14 22:16:37,455 - INFO - Environment initialized successfully
2025-11-14 22:16:37,455 - INFO - Database location: /Users/shayanfareed/Developer/dwh/data_warehouse/global_health_datawarehouse.db


### 2. DATA EXTRACTION

In [3]:
extraction_start = datetime.now()
logger.info("\nDATA EXTRACTION")
logger.info("-" * 70)

csv_path = data_dir / "global-health-statistics.csv"
if not csv_path.exists():
    raise FileNotFoundError(
        f"Required file not found: {csv_path}\n"
        f"Please place 'global-health-statistics.csv' in the /data folder"
    )

try:
    global_health_df = pd.read_csv(csv_path)
    logger.info(f"✓ CSV data loaded: {len(global_health_df):,} rows, {len(global_health_df.columns)} columns")
    
    pipeline_metadata['stages']['extraction'] = {
        'csv_rows': len(global_health_df),
        'csv_columns': len(global_health_df.columns),
        'status': 'success'
    }
except Exception as e:
    logger.error(f"✗ Failed to load CSV: {e}")
    raise

covid_df = pd.DataFrame()
try:
    response = requests.get("https://disease.sh/v3/covid-19/countries", timeout=10)
    response.raise_for_status()
    
    covid_data = response.json()
    covid_df = pd.DataFrame(covid_data)
    
    covid_columns = [
        'country', 'cases', 'todayCases', 'deaths', 'todayDeaths', 
        'recovered', 'active', 'critical', 'casesPerOneMillion',
        'deathsPerOneMillion', 'tests', 'population', 'continent'
    ]
    covid_df = covid_df[[col for col in covid_columns if col in covid_df.columns]]
    covid_df['extraction_date'] = datetime.now().strftime("%Y-%m-%d")
    
    logger.info(f"✓ COVID-19 API data fetched: {len(covid_df):,} countries")
    pipeline_metadata['stages']['extraction']['covid_countries'] = len(covid_df)
    
except Exception as e:
    logger.warning(f"⚠ COVID-19 API unavailable: {e}")
    pipeline_metadata['stages']['extraction']['covid_status'] = 'failed'

extraction_duration = (datetime.now() - extraction_start).total_seconds()
logger.info(f"Extraction completed in {extraction_duration:.2f}s")

2025-11-14 22:16:37,462 - INFO - 
DATA EXTRACTION
2025-11-14 22:16:37,462 - INFO - ----------------------------------------------------------------------
2025-11-14 22:16:38,375 - INFO - ✓ CSV data loaded: 1,000,000 rows, 22 columns
2025-11-14 22:16:40,430 - INFO - ✓ COVID-19 API data fetched: 231 countries
2025-11-14 22:16:40,431 - INFO - Extraction completed in 2.97s


### 3. DATA QUALITY ASSESSMENT

In [4]:
logger.info("\nDATA QUALITY ASSESSMENT")
logger.info("-" * 70)

quality_report = {
    'total_records': len(global_health_df),
    'total_columns': len(global_health_df.columns),
    'column_names': list(global_health_df.columns),
    'missing_values_total': int(global_health_df.isna().sum().sum()),
    'missing_by_column': global_health_df.isna().sum().to_dict(),
    'duplicate_rows': int(global_health_df.duplicated().sum()),
    'data_types': global_health_df.dtypes.astype(str).to_dict(),
    'memory_usage_mb': global_health_df.memory_usage(deep=True).sum() / 1024**2
}

numeric_cols = global_health_df.select_dtypes(include=[np.number]).columns
quality_report['numeric_summary'] = {
    col: {
        'min': float(global_health_df[col].min()),
        'max': float(global_health_df[col].max()),
        'mean': float(global_health_df[col].mean()),
        'median': float(global_health_df[col].median()),
        'std': float(global_health_df[col].std())
    }
    for col in numeric_cols
}

categorical_cols = global_health_df.select_dtypes(include=['object']).columns
quality_report['categorical_summary'] = {
    col: {
        'unique_values': int(global_health_df[col].nunique()),
        'most_common': global_health_df[col].value_counts().head(3).to_dict()
    }
    for col in categorical_cols
}

logger.info(f"Dataset Profile:")
logger.info(f"  • Records: {quality_report['total_records']:,}")
logger.info(f"  • Columns: {quality_report['total_columns']}")
logger.info(f"  • Missing values: {quality_report['missing_values_total']:,}")
logger.info(f"  • Duplicate rows: {quality_report['duplicate_rows']:,}")
logger.info(f"  • Memory usage: {quality_report['memory_usage_mb']:.2f} MB")

pipeline_metadata['stages']['quality_assessment'] = quality_report

2025-11-14 22:16:40,443 - INFO - 
DATA QUALITY ASSESSMENT
2025-11-14 22:16:40,444 - INFO - ----------------------------------------------------------------------
2025-11-14 22:16:42,039 - INFO - Dataset Profile:
2025-11-14 22:16:42,040 - INFO -   • Records: 1,000,000
2025-11-14 22:16:42,040 - INFO -   • Columns: 22
2025-11-14 22:16:42,040 - INFO -   • Missing values: 0
2025-11-14 22:16:42,040 - INFO -   • Duplicate rows: 0
2025-11-14 22:16:42,041 - INFO -   • Memory usage: 484.66 MB


### 4. DATA TRANSFORMATION

In [5]:
transformation_start = datetime.now()
logger.info("\n[PHASE 3] DATA TRANSFORMATION")
logger.info("-" * 70)

df = global_health_df.copy()

logger.info("Standardizing column names...")
df.columns = [c.strip().replace(' ', '_').replace('/', '_').lower() for c in df.columns]

initial_rows = len(df)
df = df.drop_duplicates()
removed_dupes = initial_rows - len(df)
if removed_dupes > 0:
    logger.info(f"✓ Removed {removed_dupes:,} duplicate rows")

logger.info("Cleaning string data...")
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace('nan', np.nan)

logger.info("Handling missing values...")
numeric_columns = df.select_dtypes(include=[np.number]).columns
for col in numeric_columns:
    if df[col].isna().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        logger.info(f"  • {col}: filled {df[col].isna().sum()} values with median ({median_val:.2f})")

logger.info("\nCreating dimension tables...")

dim_country = df[['country', 'per_capita_income_(usd)', 'education_index', 'urbanization_rate_(%)']].drop_duplicates(subset=['country']).reset_index(drop=True)
dim_country['country_id'] = range(1, len(dim_country) + 1)
dim_country = dim_country[['country_id', 'country', 'per_capita_income_(usd)', 'education_index', 'urbanization_rate_(%)']]
logger.info(f"  • DimCountry: {len(dim_country)} countries (with socioeconomic attributes)")

dim_disease = df[[
    'disease_name', 'disease_category', 'treatment_type', 
    'availability_of_vaccines_treatment'
]].drop_duplicates().reset_index(drop=True)
dim_disease['disease_id'] = range(1, len(dim_disease) + 1)
dim_disease = dim_disease[[
    'disease_id', 'disease_name', 'disease_category', 
    'treatment_type', 'availability_of_vaccines_treatment'
]]
logger.info(f"  • DimDisease: {len(dim_disease)} unique disease profiles")

dim_demographics = df[['age_group', 'gender']].drop_duplicates().reset_index(drop=True)
dim_demographics['demographics_id'] = range(1, len(dim_demographics) + 1)
dim_demographics = dim_demographics[['demographics_id', 'age_group', 'gender']]
logger.info(f"  • DimDemographics: {len(dim_demographics)} demographic segments")

dim_time = df[['year']].drop_duplicates().reset_index(drop=True)
dim_time['time_id'] = range(1, len(dim_time) + 1)
dim_time = dim_time[['time_id', 'year']]
dim_time = dim_time.sort_values('year').reset_index(drop=True)
logger.info(f"  • DimTime: {len(dim_time)} years ({dim_time['year'].min()}-{dim_time['year'].max()})")

if not covid_df.empty:
    dim_covid = covid_df.copy()
    dim_covid['covid_id'] = range(1, len(dim_covid) + 1)
    cols_order = ['covid_id'] + [col for col in dim_covid.columns if col != 'covid_id']
    dim_covid = dim_covid[cols_order]
    logger.info(f"  • DimCovidStats: {len(dim_covid)} countries")
else:
    dim_covid = pd.DataFrame()
    logger.warning("  • DimCovidStats: No data available")

logger.info("\nCreating fact table...")

fact_health = df.merge(dim_country, on='country', how='left')
fact_health = fact_health.merge(
    dim_disease, 
    on=['disease_name', 'disease_category', 'treatment_type', 'availability_of_vaccines_treatment'],
    how='left'
)
fact_health = fact_health.merge(dim_demographics, on=['age_group', 'gender'], how='left')
fact_health = fact_health.merge(dim_time, on='year', how='left')

if not dim_covid.empty:
    covid_lookup = dim_covid[['covid_id', 'country']].copy()
    fact_health = fact_health.merge(covid_lookup, on='country', how='left')
    logger.info(f"✓ COVID data linked: {fact_health['covid_id'].notna().sum():,} records matched")
else:
    fact_health['covid_id'] = None
    logger.warning("⚠ No COVID data available for linking")

fact_columns = [
    'country_id', 'disease_id', 'demographics_id', 'time_id', 'covid_id',
    'prevalence_rate_(%)', 'incidence_rate_(%)', 'mortality_rate_(%)',
    'population_affected', 'healthcare_access_(%)', 'doctors_per_1000',
    'hospital_beds_per_1000', 'average_treatment_cost_(usd)',
    'recovery_rate_(%)', 'dalys', 'improvement_in_5_years_(%)'
]

fact_health = fact_health[fact_columns].copy()
fact_health.insert(0, 'fact_id', range(1, len(fact_health) + 1))

logger.info(f"✓ FactHealthOutcomes: {len(fact_health):,} records")

transformation_duration = (datetime.now() - transformation_start).total_seconds()
logger.info(f"Transformation completed in {transformation_duration:.2f}s")

pipeline_metadata['stages']['transformation'] = {
    'duration_seconds': transformation_duration,
    'dim_country': len(dim_country),
    'dim_disease': len(dim_disease),
    'dim_demographics': len(dim_demographics),
    'dim_time': len(dim_time),
    'dim_covid': len(dim_covid) if not dim_covid.empty else 0,
    'fact_health': len(fact_health)
}

2025-11-14 22:16:42,049 - INFO - 
[PHASE 3] DATA TRANSFORMATION
2025-11-14 22:16:42,050 - INFO - ----------------------------------------------------------------------
2025-11-14 22:16:42,069 - INFO - Standardizing column names...
2025-11-14 22:16:42,448 - INFO - Cleaning string data...
2025-11-14 22:16:43,042 - INFO - Handling missing values...
2025-11-14 22:16:43,061 - INFO - 
Creating dimension tables...
2025-11-14 22:16:43,085 - INFO -   • DimCountry: 20 countries (with socioeconomic attributes)
2025-11-14 22:16:43,213 - INFO -   • DimDisease: 1760 unique disease profiles
2025-11-14 22:16:43,275 - INFO -   • DimDemographics: 12 demographic segments
2025-11-14 22:16:43,280 - INFO -   • DimTime: 25 years (2000-2024)
2025-11-14 22:16:43,280 - INFO -   • DimCovidStats: 231 countries
2025-11-14 22:16:43,281 - INFO - 
Creating fact table...
2025-11-14 22:16:43,960 - INFO - ✓ COVID data linked: 949,819 records matched
2025-11-14 22:16:44,015 - INFO - ✓ FactHealthOutcomes: 1,000,000 record

### 5. DATA WAREHOUSE LOADING

In [6]:
loading_start = datetime.now()
logger.info("\n[PHASE 4] DATA WAREHOUSE LOADING")
logger.info("-" * 70)

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

logger.info("Creating database schema...")

schema_sql = {
    'DimCountry': """
        CREATE TABLE IF NOT EXISTS DimCountry (
            country_id INTEGER PRIMARY KEY,
            country TEXT NOT NULL UNIQUE
        )
    """,
    'DimDisease': """
        CREATE TABLE IF NOT EXISTS DimDisease (
            disease_id INTEGER PRIMARY KEY,
            disease_name TEXT NOT NULL,
            disease_category TEXT,
            treatment_type TEXT,
            availability_of_vaccines_treatment TEXT
        )
    """,
    'DimDemographics': """
        CREATE TABLE IF NOT EXISTS DimDemographics (
            demographics_id INTEGER PRIMARY KEY,
            age_group TEXT NOT NULL,
            gender TEXT NOT NULL
        )
    """,
    'DimTime': """
        CREATE TABLE IF NOT EXISTS DimTime (
            time_id INTEGER PRIMARY KEY,
            year INTEGER NOT NULL UNIQUE
        )
    """,
    'DimCovidStats': """
        CREATE TABLE IF NOT EXISTS DimCovidStats (
            covid_id INTEGER PRIMARY KEY,
            country TEXT,
            cases INTEGER,
            todayCases INTEGER,
            deaths INTEGER,
            todayDeaths INTEGER,
            recovered INTEGER,
            active INTEGER,
            critical INTEGER,
            casesPerOneMillion REAL,
            deathsPerOneMillion REAL,
            tests INTEGER,
            population INTEGER,
            continent TEXT,
            extraction_date TEXT
        )
    """,
    'FactHealthOutcomes': """
        CREATE TABLE IF NOT EXISTS FactHealthOutcomes (
            fact_id INTEGER PRIMARY KEY,
            country_id INTEGER,
            disease_id INTEGER,
            demographics_id INTEGER,
            time_id INTEGER,
            covid_id INTEGER,
            prevalence_rate REAL,
            incidence_rate REAL,
            mortality_rate REAL,
            population_affected INTEGER,
            healthcare_access REAL,
            doctors_per_1000 REAL,
            hospital_beds_per_1000 REAL,
            average_treatment_cost REAL,
            recovery_rate REAL,
            dalys INTEGER,
            improvement_in_5_years REAL,
            per_capita_income REAL,
            education_index REAL,
            urbanization_rate REAL,
            FOREIGN KEY (country_id) REFERENCES DimCountry(country_id),
            FOREIGN KEY (disease_id) REFERENCES DimDisease(disease_id),
            FOREIGN KEY (demographics_id) REFERENCES DimDemographics(demographics_id),
            FOREIGN KEY (time_id) REFERENCES DimTime(time_id),
            FOREIGN KEY (covid_id) REFERENCES DimCovidStats(covid_id)
        )
    """
}

for table_name, sql in schema_sql.items():
    cursor.execute(sql)
logger.info("✓ Schema created successfully")

tables_to_load = {
    'DimCountry': dim_country,
    'DimDisease': dim_disease,
    'DimDemographics': dim_demographics,
    'DimTime': dim_time,
    'FactHealthOutcomes': fact_health
}

if not dim_covid.empty:
    tables_to_load['DimCovidStats'] = dim_covid

logger.info("\nLoading data into warehouse...")
for table_name, dataframe in tables_to_load.items():
    dataframe.to_sql(table_name, conn, if_exists='replace', index=False)
    row_count = cursor.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    logger.info(f"  • {table_name}: {row_count:,} rows loaded")

logger.info("\nCreating indexes...")
indexes = [
    "CREATE INDEX IF NOT EXISTS idx_fact_country ON FactHealthOutcomes(country_id)",
    "CREATE INDEX IF NOT EXISTS idx_fact_disease ON FactHealthOutcomes(disease_id)",
    "CREATE INDEX IF NOT EXISTS idx_fact_demo ON FactHealthOutcomes(demographics_id)",
    "CREATE INDEX IF NOT EXISTS idx_fact_time ON FactHealthOutcomes(time_id)",
    "CREATE INDEX IF NOT EXISTS idx_fact_covid ON FactHealthOutcomes(covid_id)",
    "CREATE INDEX IF NOT EXISTS idx_disease_name ON DimDisease(disease_name)",
    "CREATE INDEX IF NOT EXISTS idx_country_name ON DimCountry(country)",
    "CREATE INDEX IF NOT EXISTS idx_covid_country ON DimCovidStats(country)"
]

for idx_sql in indexes:
    cursor.execute(idx_sql)
logger.info("✓ Indexes created successfully")

conn.commit()

loading_duration = (datetime.now() - loading_start).total_seconds()
logger.info(f"Loading completed in {loading_duration:.2f}s")

2025-11-14 22:16:44,021 - INFO - 
[PHASE 4] DATA WAREHOUSE LOADING
2025-11-14 22:16:44,022 - INFO - ----------------------------------------------------------------------
2025-11-14 22:16:44,022 - INFO - Creating database schema...
2025-11-14 22:16:44,025 - INFO - ✓ Schema created successfully
2025-11-14 22:16:44,025 - INFO - 
Loading data into warehouse...
2025-11-14 22:16:44,027 - INFO -   • DimCountry: 20 rows loaded
2025-11-14 22:16:44,030 - INFO -   • DimDisease: 1,760 rows loaded
2025-11-14 22:16:44,032 - INFO -   • DimDemographics: 12 rows loaded
2025-11-14 22:16:44,033 - INFO -   • DimTime: 25 rows loaded
2025-11-14 22:16:45,468 - INFO -   • FactHealthOutcomes: 1,000,000 rows loaded
2025-11-14 22:16:45,472 - INFO -   • DimCovidStats: 231 rows loaded
2025-11-14 22:16:45,473 - INFO - 
Creating indexes...
2025-11-14 22:16:46,616 - INFO - ✓ Indexes created successfully
2025-11-14 22:16:46,616 - INFO - Loading completed in 2.59s


### 6. DATA VALIDATION & ANALYTICS

In [7]:
logger.info("\nDATA VALIDATION & ANALYTICS")
logger.info("-" * 70)

validation_queries = {
    'Total Records': "SELECT COUNT(*) as count FROM FactHealthOutcomes",
    'Countries': "SELECT COUNT(*) as count FROM DimCountry",
    'Diseases': "SELECT COUNT(*) as count FROM DimDisease",
    'Years Range': "SELECT MIN(year) as min_year, MAX(year) as max_year FROM DimTime"
}

logger.info("Database validation results:")
for name, query in validation_queries.items():
    result = pd.read_sql_query(query, conn)
    logger.info(f" • {name}: {result.iloc[0].to_dict()}")

logger.info("\nExecuting analytical queries...")

analytical_queries = {
    'Top 10 Diseases by Average Mortality Rate': """
        SELECT 
            d.disease_name,
            d.disease_category,
            ROUND(AVG(f.[mortality_rate_(%)]), 2) AS avg_mortality_rate,
            COUNT(*) AS record_count
        FROM FactHealthOutcomes f
        JOIN DimDisease d ON f.disease_id = d.disease_id
        GROUP BY d.disease_name, d.disease_category
        ORDER BY avg_mortality_rate DESC
        LIMIT 10
    """,

    'Healthcare Metrics by Country': """
        SELECT 
            c.country,
            ROUND(AVG(f.[healthcare_access_(%)]), 2) AS avg_healthcare_access,
            ROUND(AVG(f.doctors_per_1000), 2) AS avg_doctors_per_1000,
            ROUND(AVG(f.hospital_beds_per_1000), 2) AS avg_hospital_beds,
            ROUND(AVG(f.[recovery_rate_(%)]), 2) AS avg_recovery_rate
        FROM FactHealthOutcomes f
        JOIN DimCountry c ON f.country_id = c.country_id
        GROUP BY c.country
        ORDER BY avg_healthcare_access DESC
        LIMIT 10
    """,

    'Disease Burden (DALYs) by Demographics': """
        SELECT 
            dd.age_group,
            dd.gender,
            ROUND(AVG(f.dalys), 0) AS avg_dalys,
            COUNT(*) AS cases
        FROM FactHealthOutcomes f
        JOIN DimDemographics dd ON f.demographics_id = dd.demographics_id
        GROUP BY dd.age_group, dd.gender
        ORDER BY avg_dalys DESC
    """,

    'Trend Analysis - Health Improvements Over Years': """
        SELECT 
            t.year,
            ROUND(AVG(f.[improvement_in_5_years_(%)]), 2) AS avg_improvement,
            ROUND(AVG(f.[recovery_rate_(%)]), 2) AS avg_recovery_rate,
            ROUND(AVG(f.[mortality_rate_(%)]), 2) AS avg_mortality_rate
        FROM FactHealthOutcomes f
        JOIN DimTime t ON f.time_id = t.time_id
        GROUP BY t.year
        ORDER BY t.year
    """,

    'COVID-19 Impact Analysis (API Data Integration)': """
        SELECT 
            c.country,
            cv.continent,
            cv.cases AS total_covid_cases,
            cv.deaths AS total_covid_deaths,
            cv.recovered AS total_recovered,
            ROUND(CAST(cv.deaths AS FLOAT) / cv.cases * 100, 2) AS covid_mortality_rate,
            ROUND(AVG(f.[healthcare_access_(%)]), 2) AS avg_healthcare_access,
            ROUND(AVG(f.doctors_per_1000), 2) AS avg_doctors_per_1000,
            cv.population
        FROM FactHealthOutcomes f
        JOIN DimCountry c ON f.country_id = c.country_id
        JOIN DimCovidStats cv ON f.covid_id = cv.covid_id
        WHERE cv.cases > 0
        GROUP BY c.country, cv.continent, cv.cases, cv.deaths, cv.recovered, cv.population
        ORDER BY cv.cases DESC
        LIMIT 20
    """,

    'COVID vs Historical Disease Mortality Comparison': """
        SELECT 
            c.country,
            ROUND(AVG(CASE WHEN d.disease_name = 'COVID-19' THEN f.[mortality_rate_(%)] END), 2) AS covid_mortality_historical,
            ROUND(AVG(CASE WHEN d.disease_name != 'COVID-19' THEN f.[mortality_rate_(%)] END), 2) AS other_diseases_avg_mortality,
            cv.deaths AS current_covid_deaths,
            cv.cases AS current_covid_cases,
            ROUND(CAST(cv.deaths AS FLOAT) / NULLIF(cv.cases, 0) * 100, 2) AS current_covid_mortality_rate
        FROM FactHealthOutcomes f
        JOIN DimCountry c ON f.country_id = c.country_id
        JOIN DimDisease d ON f.disease_id = d.disease_id
        LEFT JOIN DimCovidStats cv ON f.covid_id = cv.covid_id
        WHERE cv.cases > 10000
        GROUP BY c.country, cv.deaths, cv.cases
        HAVING covid_mortality_historical IS NOT NULL
        ORDER BY current_covid_cases DESC
        LIMIT 15
    """,

    'Healthcare Infrastructure vs COVID Outcomes': """
        SELECT 
            cv.continent,
            COUNT(DISTINCT c.country) AS countries_count,
            ROUND(AVG(f.doctors_per_1000), 2) AS avg_doctors,
            ROUND(AVG(f.hospital_beds_per_1000), 2) AS avg_beds,
            ROUND(AVG(f.[healthcare_access_(%)]), 2) AS avg_healthcare_access,
            SUM(cv.cases) AS total_covid_cases,
            SUM(cv.deaths) AS total_covid_deaths,
            ROUND(CAST(SUM(cv.deaths) AS FLOAT) / NULLIF(SUM(cv.cases), 0) * 100, 2) AS continent_mortality_rate
        FROM FactHealthOutcomes f
        JOIN DimCountry c ON f.country_id = c.country_id
        JOIN DimCovidStats cv ON f.covid_id = cv.covid_id
        WHERE cv.continent IS NOT NULL
        GROUP BY cv.continent
        ORDER BY total_covid_cases DESC
    """
}

analytical_results = {}

for name, query in analytical_queries.items():
    try:
        result = pd.read_sql_query(query, conn)
        analytical_results[name] = result
        logger.info(f"\n{name}:")
        print(result.to_string(index=False))
    except Exception as e:
        logger.error(f"✗ Query failed - {name}: {e}")
logger.info("\nCreating unified dataset for BI dashboard...")

bi_view_sql = """
SELECT 
    f.fact_id,
    c.country,
    c.[per_capita_income_(usd)],
    c.education_index,
    c.[urbanization_rate_(%)],
    d.disease_name,
    d.disease_category,
    dd.age_group,
    dd.gender,
    t.year,
    f.[prevalence_rate_(%)],
    f.[incidence_rate_(%)],
    f.[mortality_rate_(%)],
    f.population_affected,
    f.[healthcare_access_(%)],
    f.doctors_per_1000,
    f.hospital_beds_per_1000,
    f.[average_treatment_cost_(usd)],
    f.[recovery_rate_(%)],
    f.dalys,
    f.[improvement_in_5_years_(%)],
    cv.cases              AS covid_total_cases,
    cv.deaths             AS covid_total_deaths,
    cv.recovered          AS covid_recovered,
    cv.population         AS covid_population,
    ROUND(CAST(cv.deaths AS FLOAT) / NULLIF(cv.cases, 0) * 100, 2) AS covid_mortality_rate
FROM FactHealthOutcomes f
JOIN DimCountry c        ON f.country_id = c.country_id
JOIN DimDisease d        ON f.disease_id = d.disease_id
JOIN DimDemographics dd  ON f.demographics_id = dd.demographics_id
JOIN DimTime t           ON f.time_id = t.time_id
LEFT JOIN DimCovidStats cv ON f.covid_id = cv.covid_id
"""

bi_df = pd.read_sql_query(bi_view_sql, conn)

2025-11-14 22:16:46,623 - INFO - 
DATA VALIDATION & ANALYTICS
2025-11-14 22:16:46,623 - INFO - ----------------------------------------------------------------------
2025-11-14 22:16:46,624 - INFO - Database validation results:
2025-11-14 22:16:46,627 - INFO -  • Total Records: {'count': 1000000}
2025-11-14 22:16:46,627 - INFO -  • Countries: {'count': 20}
2025-11-14 22:16:46,627 - INFO -  • Diseases: {'count': 1760}
2025-11-14 22:16:46,628 - INFO -  • Years Range: {'min_year': 2000, 'max_year': 2024}
2025-11-14 22:16:46,628 - INFO - 
Executing analytical queries...
2025-11-14 22:16:48,115 - INFO - 
Top 10 Diseases by Average Mortality Rate:


       disease_name disease_category  avg_mortality_rate  record_count
Alzheimer's Disease   Cardiovascular                5.14          4538
             Cancer        Metabolic                5.14          4612
            Malaria        Metabolic                5.14          4573
             Rabies   Cardiovascular                5.14          4657
Alzheimer's Disease          Genetic                5.12          4378
             Asthma        Metabolic                5.12          4654
             Dengue       Infectious                5.12          4538
          Hepatitis        Metabolic                5.12          4571
               Zika          Chronic                5.12          4652
Alzheimer's Disease            Viral                5.11          4685


2025-11-14 22:16:48,630 - INFO - 
Healthcare Metrics by Country:


     country  avg_healthcare_access  avg_doctors_per_1000  avg_hospital_beds  avg_recovery_rate
      Russia                  75.17                  2.75               5.23              74.49
South Africa                  75.10                  2.74               5.25              74.46
     Nigeria                  75.07                  2.74               5.25              74.46
       Japan                  75.06                  2.75               5.26              74.42
       India                  75.03                  2.75               5.25              74.46
   Indonesia                  75.03                  2.75               5.23              74.60
     Germany                  75.02                  2.74               5.25              74.55
       China                  75.00                  2.75               5.24              74.53
      France                  74.98                  2.75               5.24              74.46
   Argentina                  74.97     

2025-11-14 22:16:49,391 - INFO - 
Disease Burden (DALYs) by Demographics:


...

     country         continent  total_covid_cases  total_covid_deaths  total_recovered  covid_mortality_rate  avg_healthcare_access  avg_doctors_per_1000  population
         USA     North America          111820082             1219487        109814428                  1.09                  74.96                  2.76   334805269
       India              Asia           45035393              533570                0                  1.18                  75.03                  2.75  1406631776
      France            Europe           40138560              167642         39970918                  0.42                  74.98                  2.75    65584518
     Germany            Europe           38828995              183027         38240600                  0.47                  75.02                  2.74    83883596
      Brazil     South America           38743918              711380         36249161                  1.84                  74.97                  2.75   215353593
    

2025-11-14 22:16:53,078 - INFO - 
COVID vs Historical Disease Mortality Comparison:


  country  covid_mortality_historical  other_diseases_avg_mortality  current_covid_deaths  current_covid_cases  current_covid_mortality_rate
      USA                        5.03                          5.04               1219487            111820082                          1.09
    India                        5.02                          5.05                533570             45035393                          1.18
   France                        5.04                          5.07                167642             40138560                          0.42
  Germany                        4.96                          5.07                183027             38828995                          0.47
   Brazil                        5.18                          5.06                711380             38743918                          1.84
    Japan                        5.04                          5.06                 74694             33803572                          0.22
    Italy    

2025-11-14 22:16:54,044 - INFO - 
Healthcare Infrastructure vs COVID Outcomes:
2025-11-14 22:16:54,048 - INFO - 
Creating unified dataset for BI dashboard...


        continent  countries_count  avg_doctors  avg_beds  avg_healthcare_access  total_covid_cases  total_covid_deaths  continent_mortality_rate
           Europe                5         2.75      5.24                  75.02      7752461742866         59335502943                      0.77
    North America                3         2.75      5.24                  74.93      6214900781846         80601381147                      1.30
             Asia                6         2.75      5.25                  74.99      5190090584464         44175549638                      0.85
    South America                2         2.75      5.26                  74.97      2429465276976         41861958178                      1.72
Australia-Oceania                1         2.74      5.23                  74.88       592100102232          1219552542                      0.21
           Africa                2         2.74      5.25                  75.09       218858037552          5329503890     

### 7. EXPORT DATASETS

In [8]:
logger.info("\nEXPORTING DATASETS")
logger.info("-" * 70)

comprehensive_view_sql = """
    SELECT 
        f.fact_id,
        c.country,
        c.[per_capita_income_(usd)],
        c.education_index,
        c.[urbanization_rate_(%)],
        d.disease_name,
        d.disease_category,
        d.treatment_type,
        d.availability_of_vaccines_treatment,
        dd.age_group,
        dd.gender,
        t.year,
        f.[prevalence_rate_(%)],
        f.[incidence_rate_(%)],
        f.[mortality_rate_(%)],
        f.population_affected,
        f.[healthcare_access_(%)],
        f.doctors_per_1000,
        f.hospital_beds_per_1000,
        f.[average_treatment_cost_(usd)],
        f.[recovery_rate_(%)],
        f.dalys,
        f.[improvement_in_5_years_(%)],
        -- COVID API data integration
        cv.cases AS covid_total_cases,
        cv.deaths AS covid_total_deaths,
        cv.recovered AS covid_recovered,
        cv.active AS covid_active_cases,
        cv.critical AS covid_critical_cases,
        cv.casesPerOneMillion AS covid_cases_per_million,
        cv.deathsPerOneMillion AS covid_deaths_per_million,
        cv.continent,
        cv.population AS country_population
    FROM FactHealthOutcomes f
    JOIN DimCountry c ON f.country_id = c.country_id
    JOIN DimDisease d ON f.disease_id = d.disease_id
    JOIN DimDemographics dd ON f.demographics_id = dd.demographics_id
    JOIN DimTime t ON f.time_id = t.time_id
    LEFT JOIN DimCovidStats cv ON f.covid_id = cv.covid_id
"""

comprehensive_df = pd.read_sql_query(comprehensive_view_sql, conn)
comprehensive_path = exports_dir / "global_health_analytics_full.csv"
comprehensive_df.to_csv(comprehensive_path, index=False)
logger.info(f"✓ Comprehensive dataset: {comprehensive_path} ({len(comprehensive_df):,} rows)")

ml_features_df = comprehensive_df[[
    'country', 'disease_category', 'age_group', 'gender', 'year',
    'education_index', 'per_capita_income_(usd)', 'healthcare_access_(%)',
    'doctors_per_1000', 'hospital_beds_per_1000', 'urbanization_rate_(%)',
    'recovery_rate_(%)', 'mortality_rate_(%)', 'dalys',
    'covid_total_cases', 'covid_total_deaths',
    'covid_cases_per_million', 'covid_deaths_per_million',
    'continent', 'country_population'
]].copy()

ml_features_df.fillna({
    'covid_total_cases': 0,
    'covid_total_deaths': 0,
    'covid_cases_per_million': 0,
    'covid_deaths_per_million': 0,
    'continent': 'Unknown',
    'country_population': 0
}, inplace=True)

ml_path = exports_dir / "ml_health_features.csv"
ml_features_df.to_csv(ml_path, index=False)
logger.info(f"✓ ML features dataset: {ml_path} ({len(ml_features_df):,} rows)")

logger.info("\nCreating Unified BI dataset export...")

bi_df = comprehensive_df[[
    'country', 'continent', 'year', 'disease_name', 'disease_category',
    'age_group', 'gender',
    'per_capita_income_(usd)', 'education_index', 'urbanization_rate_(%)',
    'healthcare_access_(%)', 'doctors_per_1000', 'hospital_beds_per_1000',
    'recovery_rate_(%)', 'mortality_rate_(%)', 'dalys', 'improvement_in_5_years_(%)',
    'covid_total_cases', 'covid_total_deaths', 'covid_recovered',
    'covid_cases_per_million', 'covid_deaths_per_million',
    'country_population'
]].copy()

bi_df['covid_mortality_rate'] = round(
    (bi_df['covid_total_deaths'] / bi_df['covid_total_cases']) * 100, 2
).replace([np.inf, -np.inf], np.nan).fillna(0)

bi_path = exports_dir / "global_health_bi_dashboard.csv"
bi_df.to_csv(bi_path, index=False)
logger.info(f"✓ Unified BI dataset: {bi_path} ({len(bi_df):,} rows)")


2025-11-14 22:17:01,491 - INFO - 
EXPORTING DATASETS
2025-11-14 22:17:01,492 - INFO - ----------------------------------------------------------------------
2025-11-14 22:17:15,435 - INFO - ✓ Comprehensive dataset: /Users/shayanfareed/Developer/dwh/data_warehouse/exports/global_health_analytics_full.csv (1,000,000 rows)
2025-11-14 22:17:19,755 - INFO - ✓ ML features dataset: /Users/shayanfareed/Developer/dwh/data_warehouse/exports/ml_health_features.csv (1,000,000 rows)
2025-11-14 22:17:19,756 - INFO - 
Creating Unified BI dataset export...
2025-11-14 22:17:25,306 - INFO - ✓ Unified BI dataset: /Users/shayanfareed/Developer/dwh/data_warehouse/exports/global_health_bi_dashboard.csv (1,000,000 rows)


### 8. MACHINE LEARNING

In [ ]:
logger.info("\MACHINE LEARNING")
logger.info("-" * 70)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error,
                             silhouette_score, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score)
warnings.filterwarnings('ignore')

ml_start = datetime.now()

logger.info("Preparing enhanced ML dataset with feature engineering...")
ml_data = ml_features_df.copy()

ml_data = ml_data.dropna(subset=['mortality_rate_(%)', 'recovery_rate_(%)', 'dalys'])

logger.info("Creating engineered features...")

ml_data['healthcare_score'] = (
    ml_data['healthcare_access_(%)'] * 0.4 +
    ml_data['doctors_per_1000'] * 10 +
    ml_data['hospital_beds_per_1000'] * 10
) / 3

ml_data['socioeconomic_index'] = (
    ml_data['per_capita_income_(usd)'] / 100000 * 0.5 +
    ml_data['education_index'] * 100 * 0.3 +
    ml_data['urbanization_rate_(%)'] * 0.2
)

ml_data['covid_impact_score'] = np.where(
    ml_data['covid_cases_per_million'] > 0,
    np.log1p(ml_data['covid_cases_per_million']) + 
    np.log1p(ml_data['covid_deaths_per_million']) * 2,
    0
)

ml_data['years_since_2000'] = ml_data['year'] - 2000
ml_data['decade'] = (ml_data['year'] // 10) * 10

ml_data['doctor_bed_ratio'] = ml_data['doctors_per_1000'] / (ml_data['hospital_beds_per_1000'] + 0.1)

ml_data['mortality_recovery_ratio'] = ml_data['mortality_rate_(%)'] / (ml_data['recovery_rate_(%)'] + 1)

logger.info(f"✓ Created 7 engineered features")

label_encoders = {}
categorical_cols = ['country', 'disease_category', 'age_group', 'gender', 'continent']

for col in categorical_cols:
    if col in ml_data.columns:
        le = LabelEncoder()
        ml_data[f'{col}_encoded'] = le.fit_transform(ml_data[col].astype(str))
        label_encoders[col] = le

logger.info(f"✓ Encoded {len(categorical_cols)} categorical features")
logger.info(f"✓ Final dataset: {len(ml_data):,} samples with {len(ml_data.columns)} features")

logger.info("\n[ML Model 1] Enhanced Mortality Rate Prediction")
logger.info("-" * 70)

mortality_features = [
    'healthcare_score', 'socioeconomic_index', 'covid_impact_score',
    'years_since_2000', 'doctor_bed_ratio',
    'healthcare_access_(%)', 'doctors_per_1000', 'hospital_beds_per_1000',
    'per_capita_income_(usd)', 'education_index', 'urbanization_rate_(%)',
    'recovery_rate_(%)', 'dalys',
    'country_encoded', 'disease_category_encoded', 
    'age_group_encoded', 'gender_encoded'
]

mortality_data = ml_data[mortality_features + ['mortality_rate_(%)']].copy()

Q1 = mortality_data['mortality_rate_(%)'].quantile(0.25)
Q3 = mortality_data['mortality_rate_(%)'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
mortality_data = mortality_data[
    (mortality_data['mortality_rate_(%)'] >= lower_bound) & 
    (mortality_data['mortality_rate_(%)'] <= upper_bound)
]

X_mortality = mortality_data[mortality_features]
y_mortality = mortality_data['mortality_rate_(%)']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_mortality, y_mortality, test_size=0.2, random_state=42
)

scaler_mortality = RobustScaler()
X_train_m_scaled = scaler_mortality.fit_transform(X_train_m)
X_test_m_scaled = scaler_mortality.transform(X_test_m)

logger.info("Training and comparing multiple models...")

rf_mortality = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=50,
    min_samples_leaf=20,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf_mortality.fit(X_train_m_scaled, y_train_m)
y_pred_rf = rf_mortality.predict(X_test_m_scaled)

ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_m_scaled, y_train_m)
y_pred_ridge = ridge.predict(X_test_m_scaled)

from sklearn.ensemble import GradientBoostingRegressor
gb_regressor = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)
gb_regressor.fit(X_train_m_scaled, y_train_m)
y_pred_gb = gb_regressor.predict(X_test_m_scaled)

models_comparison = []

for name, y_pred in [('Random Forest', y_pred_rf), ('Ridge', y_pred_ridge), ('Gradient Boosting', y_pred_gb)]:
    mse = mean_squared_error(y_test_m, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_m, y_pred)
    r2 = r2_score(y_test_m, y_pred)
    
    models_comparison.append({
        'model': name,
        'rmse': rmse,
        'mae': mae,
        'r2_score': r2
    })
    
    logger.info(f"\n{name} Performance:")
    logger.info(f"  • RMSE: {rmse:.4f}")
    logger.info(f"  • MAE: {mae:.4f}")
    logger.info(f"  • R² Score: {r2:.4f}")

best_model_idx = max(range(len(models_comparison)), key=lambda i: models_comparison[i]['r2_score'])
best_model_name = models_comparison[best_model_idx]['model']
logger.info(f"\n✓ Best Model: {best_model_name}")
logger.info(f"  Training samples: {len(X_train_m):,}")
logger.info(f"  Test samples: {len(X_test_m):,}")

feature_importance = pd.DataFrame({
    'feature': mortality_features,
    'importance': rf_mortality.feature_importances_
}).sort_values('importance', ascending=False)

logger.info(f"\nTop 10 Features Affecting Mortality:")
for idx, row in feature_importance.head(10).iterrows():
    logger.info(f"  • {row['feature']}: {row['importance']:.4f}")

mortality_predictions = pd.DataFrame({
    'actual_mortality': y_test_m.values,
    'predicted_rf': y_pred_rf,
    'predicted_ridge': y_pred_ridge,
    'predicted_gb': y_pred_gb,
    'best_prediction': y_pred_rf if best_model_name == 'Random Forest' else 
                       (y_pred_ridge if best_model_name == 'Ridge' else y_pred_gb),
    'error': y_test_m.values - (y_pred_rf if best_model_name == 'Random Forest' else 
                                 (y_pred_ridge if best_model_name == 'Ridge' else y_pred_gb))
})
mortality_pred_path = exports_dir / "ml_enhanced_mortality_predictions.csv"
mortality_predictions.to_csv(mortality_pred_path, index=False)
logger.info(f"✓ Predictions saved: {mortality_pred_path}")

comparison_df = pd.DataFrame(models_comparison)
comparison_path = exports_dir / "ml_model_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
logger.info(f"✓ Model comparison saved: {comparison_path}")

logger.info("\n[ML Model 2] Multi-Class Disease Severity Classification")
logger.info("-" * 70)

mortality_terciles = ml_data['mortality_rate_(%)'].quantile([0.33, 0.67])
ml_data['severity'] = pd.cut(
    ml_data['mortality_rate_(%)'],
    bins=[-np.inf, mortality_terciles.iloc[0], mortality_terciles.iloc[1], np.inf],
    labels=['Low', 'Medium', 'High']
)

classification_features = [
    'healthcare_score', 'socioeconomic_index', 'covid_impact_score',
    'years_since_2000', 'doctor_bed_ratio', 'mortality_recovery_ratio',
    'healthcare_access_(%)', 'doctors_per_1000', 'hospital_beds_per_1000',
    'recovery_rate_(%)', 'dalys', 'education_index',
    'country_encoded', 'disease_category_encoded', 
    'age_group_encoded', 'gender_encoded'
]

class_data = ml_data[classification_features + ['severity']].dropna()

severity_encoder = LabelEncoder()
y_severity = severity_encoder.fit_transform(class_data['severity'])
X_class = class_data[classification_features]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_severity, test_size=0.2, random_state=42, stratify=y_severity
)

scaler_class = RobustScaler()
X_train_c_scaled = scaler_class.fit_transform(X_train_c)
X_test_c_scaled = scaler_class.transform(X_test_c)

rf_classifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=50,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_classifier.fit(X_train_c_scaled, y_train_c)

y_pred_c = rf_classifier.predict(X_test_c_scaled)
y_pred_proba = rf_classifier.predict_proba(X_test_c_scaled)

accuracy = accuracy_score(y_test_c, y_pred_c)
precision = precision_score(y_test_c, y_pred_c, average='weighted')
recall = recall_score(y_test_c, y_pred_c, average='weighted')
f1 = f1_score(y_test_c, y_pred_c, average='weighted')

logger.info(f"Multi-Class Classification Performance:")
logger.info(f"  • Accuracy: {accuracy:.4f}")
logger.info(f"  • Weighted Precision: {precision:.4f}")
logger.info(f"  • Weighted Recall: {recall:.4f}")
logger.info(f"  • Weighted F1-Score: {f1:.4f}")
logger.info(f"  • Training samples: {len(X_train_c):,}")
logger.info(f"  • Test samples: {len(X_test_c):,}")

conf_matrix = confusion_matrix(y_test_c, y_pred_c)
logger.info(f"\nConfusion Matrix:")
logger.info(f"  Severity Levels: {severity_encoder.classes_}")
logger.info(f"\n{conf_matrix}")

logger.info(f"\nPer-Class Performance:")
for i, severity_level in enumerate(severity_encoder.classes_):
    mask = y_test_c == i
    if mask.sum() > 0:
        class_acc = (y_pred_c[mask] == i).sum() / mask.sum()
        logger.info(f"  • {severity_level}: {class_acc:.4f} accuracy ({mask.sum():,} samples)")

class_results = pd.DataFrame({
    'actual_severity': severity_encoder.inverse_transform(y_test_c),
    'predicted_severity': severity_encoder.inverse_transform(y_pred_c),
    'probability_low': y_pred_proba[:, 0],
    'probability_medium': y_pred_proba[:, 1],
    'probability_high': y_pred_proba[:, 2],
    'correct_prediction': y_test_c == y_pred_c
})
class_results_path = exports_dir / "ml_enhanced_severity_classification.csv"
class_results.to_csv(class_results_path, index=False)
logger.info(f"✓ Classification results saved: {class_results_path}")

logger.info("\n[ML Model 3] Advanced Country Health Profile Clustering")
logger.info("-" * 70)

country_metrics = ml_data.groupby('country').agg({
    'healthcare_score': 'mean',
    'socioeconomic_index': 'mean',
    'covid_impact_score': 'mean',
    'healthcare_access_(%)': 'mean',
    'doctors_per_1000': 'mean',
    'hospital_beds_per_1000': 'mean',
    'mortality_rate_(%)': 'mean',
    'recovery_rate_(%)': 'mean',
    'dalys': 'mean',
    'per_capita_income_(usd)': 'mean',
    'education_index': 'mean',
    'covid_deaths_per_million': 'mean',
    'country_population': 'first'
}).reset_index()

cluster_features = [
    'healthcare_score', 'socioeconomic_index', 'covid_impact_score',
    'mortality_rate_(%)', 'recovery_rate_(%)', 'dalys'
]

X_cluster = country_metrics[cluster_features].fillna(0)

scaler_cluster = RobustScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

inertias = []
silhouette_scores = []
k_range = range(2, 9)

logger.info("Finding optimal number of clusters...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    kmeans.fit(X_cluster_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster_scaled, kmeans.labels_))

optimal_k = silhouette_scores.index(max(silhouette_scores)) + 2
logger.info(f"Optimal number of clusters: {optimal_k}")
logger.info(f"Silhouette Score: {max(silhouette_scores):.4f}")

kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=20, max_iter=500)
country_metrics['cluster'] = kmeans_final.fit_predict(X_cluster_scaled)

logger.info(f"\nDetailed Cluster Analysis:")
for cluster_id in range(optimal_k):
    cluster_data = country_metrics[country_metrics['cluster'] == cluster_id]
    
    logger.info(f"\n{'='*60}")
    logger.info(f"Cluster {cluster_id + 1}: {len(cluster_data)} countries")
    logger.info(f"{'='*60}")
    logger.info(f"Countries: {', '.join(cluster_data['country'].values)}")
    logger.info(f"\nHealth Metrics:")
    logger.info(f"  • Healthcare Score: {cluster_data['healthcare_score'].mean():.2f}")
    logger.info(f"  • Socioeconomic Index: {cluster_data['socioeconomic_index'].mean():.2f}")
    logger.info(f"  • Avg Mortality Rate: {cluster_data['mortality_rate_(%)'].mean():.2f}%")
    logger.info(f"  • Avg Recovery Rate: {cluster_data['recovery_rate_(%)'].mean():.2f}%")
    logger.info(f"  • Avg DALYs: {cluster_data['dalys'].mean():.0f}")
    logger.info(f"  • COVID Impact Score: {cluster_data['covid_impact_score'].mean():.2f}")
    logger.info(f"  • Avg Per Capita Income: ${cluster_data['per_capita_income_(usd)'].mean():.2f}")

cluster_results_path = exports_dir / "ml_enhanced_country_clusters.csv"
country_metrics.to_csv(cluster_results_path, index=False)
logger.info(f"\n✓ Clustering results saved: {cluster_results_path}")

centroids = pd.DataFrame(
    scaler_cluster.inverse_transform(kmeans_final.cluster_centers_),
    columns=cluster_features
)
centroids['cluster'] = range(optimal_k)
centroids_path = exports_dir / "ml_cluster_centroids.csv"
centroids.to_csv(centroids_path, index=False)
logger.info(f"✓ Cluster centroids saved: {centroids_path}")

logger.info("\n[ML Model 4] Enhanced Principal Component Analysis")
logger.info("-" * 70)

pca = PCA(n_components=min(len(cluster_features), 5))
pca_features = pca.fit_transform(X_cluster_scaled)

explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

logger.info("PCA Results:")
for i, (ev, cv) in enumerate(zip(explained_variance, cumulative_variance), 1):
    logger.info(f"  • PC{i}: {ev:.4f} variance explained ({cv:.4f} cumulative)")

logger.info(f"\n✓ {cumulative_variance[1]:.2%} of variance explained by first 2 components")
logger.info(f"✓ {cumulative_variance[-1]:.2%} of total variance captured")

components_df = pd.DataFrame(
    pca.components_,
    columns=cluster_features,
    index=[f'PC{i+1}' for i in range(len(pca.components_))]
)

logger.info(f"\nPrincipal Component Loadings (Top Contributors):")
for i in range(min(3, len(components_df))):
    logger.info(f"\nPC{i+1} (Top 3 features):")
    top_features = components_df.iloc[i].abs().sort_values(ascending=False).head(3)
    for feat, val in top_features.items():
        logger.info(f"  • {feat}: {components_df.iloc[i][feat]:.4f}")

pca_df = pd.DataFrame(
    pca_features,
    columns=[f'PC{i+1}' for i in range(pca_features.shape[1])]
)
pca_df['country'] = country_metrics['country'].values
pca_df['cluster'] = country_metrics['cluster'].values

for metric in ['healthcare_score', 'socioeconomic_index', 'mortality_rate_(%)']:
    pca_df[metric] = country_metrics[metric].values

pca_path = exports_dir / "ml_enhanced_pca_results.csv"
pca_df.to_csv(pca_path, index=False)
logger.info(f"✓ PCA results saved: {pca_path}")

loadings_path = exports_dir / "ml_pca_component_loadings.csv"
components_df.to_csv(loadings_path)
logger.info(f"✓ Component loadings saved: {loadings_path}")

logger.info("\n[ML Model 5] Time Series Trend Analysis")
logger.info("-" * 70)

yearly_trends = ml_data.groupby('year').agg({
    'mortality_rate_(%)': 'mean',
    'recovery_rate_(%)': 'mean',
    'healthcare_access_(%)': 'mean',
    'healthcare_score': 'mean',
    'socioeconomic_index': 'mean'
}).reset_index()

for col in yearly_trends.columns:
    if col != 'year':
        yearly_trends[f'{col}_change'] = yearly_trends[col].pct_change() * 100

logger.info("5-Year Trend Summary (2020-2024):")
recent_trends = yearly_trends[yearly_trends['year'] >= 2020]
for col in ['mortality_rate_(%)', 'recovery_rate_(%)', 'healthcare_score']:
    trend_direction = "↑ Improving" if recent_trends[col].iloc[-1] > recent_trends[col].iloc[0] else "↓ Declining"
    change_pct = ((recent_trends[col].iloc[-1] - recent_trends[col].iloc[0]) / recent_trends[col].iloc[0]) * 100
    logger.info(f"  • {col}: {trend_direction} ({change_pct:+.2f}%)")

trends_path = exports_dir / "ml_yearly_health_trends.csv"
yearly_trends.to_csv(trends_path, index=False)
logger.info(f"✓ Trend analysis saved: {trends_path}")

logger.info("\n[Enhanced ML Summary Report]")
logger.info("-" * 70)

ml_duration = (datetime.now() - ml_start).total_seconds()

best_model_metrics = models_comparison[best_model_idx]

ml_summary = {
    'execution_time_seconds': ml_duration,
    'total_samples_analyzed': len(ml_data),
    'engineered_features_created': 7,
    'models_trained': 5,
    
    'mortality_prediction': {
        'best_model': best_model_name,
        'rmse': float(best_model_metrics['rmse']),
        'mae': float(best_model_metrics['mae']),
        'r2_score': float(best_model_metrics['r2_score']),
        'training_samples': len(X_train_m),
        'test_samples': len(X_test_m),
        'top_feature': feature_importance.iloc[0]['feature'],
        'top_feature_importance': float(feature_importance.iloc[0]['importance'])
    },
    
    'severity_classification': {
        'model': 'Random Forest Classifier (Multi-Class)',
        'classes': list(severity_encoder.classes_),
        'accuracy': float(accuracy),
        'weighted_precision': float(precision),
        'weighted_recall': float(recall),
        'weighted_f1_score': float(f1),
        'training_samples': len(X_train_c),
        'test_samples': len(X_test_c)
    },
    
    'clustering': {
        'model': 'K-Means (Enhanced)',
        'optimal_clusters': int(optimal_k),
        'silhouette_score': float(max(silhouette_scores)),
        'countries_analyzed': len(country_metrics),
        'features_used': cluster_features
    },
    
    'pca': {
        'components': int(pca.n_components_),
        'variance_explained_2_components': float(cumulative_variance[1]),
        'total_variance_explained': float(cumulative_variance[-1]),
        'most_important_pc1_feature': components_df.iloc[0].abs().idxmax()
    },
    
    'trend_analysis': {
        'years_analyzed': len(yearly_trends),
        'metrics_tracked': 5,
        'recent_mortality_trend': 'improving' if recent_trends['mortality_rate_(%)'].iloc[-1] < recent_trends['mortality_rate_(%)'].iloc[0] else 'worsening'
    }
}

pipeline_metadata['stages']['machine_learning'] = ml_summary

2025-11-14 22:17:25,336 - INFO - \MACHINE LEARNING
2025-11-14 22:17:25,337 - INFO - ----------------------------------------------------------------------
2025-11-14 22:17:27,655 - INFO - Preparing enhanced ML dataset with feature engineering...
2025-11-14 22:17:28,106 - INFO - Creating engineered features...
2025-11-14 22:17:28,169 - INFO - ✓ Created 7 engineered features
2025-11-14 22:17:28,834 - INFO - ✓ Encoded 5 categorical features
2025-11-14 22:17:28,835 - INFO - ✓ Final dataset: 1,000,000 samples with 32 features
2025-11-14 22:17:28,835 - INFO - 
[ML Model 1] Enhanced Mortality Rate Prediction
2025-11-14 22:17:28,835 - INFO - ----------------------------------------------------------------------
2025-11-14 22:17:29,375 - INFO - Training and comparing multiple models...


### 10. PIPELINE COMPLETION

In [ ]:
conn.close()
total_duration = (datetime.now() - extraction_start).total_seconds()

logger.info("\n" + "=" * 70)
logger.info("ETL PIPELINE COMPLETED SUCCESSFULLY")
logger.info("=" * 70)
logger.info(f"Total execution time: {total_duration:.2f}s")
logger.info(f"Database: {db_path}")
logger.info(f"Exports: {exports_dir}")
logger.info(f"Logs: {logs_dir}")

pipeline_metadata['total_duration_seconds'] = total_duration
pipeline_metadata['status'] = 'completed'
metadata_path = logs_dir / f"pipeline_metadata_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(metadata_path, 'w') as f:
    json.dump(pipeline_metadata, f, indent=2, default=str)
logger.info(f"Pipeline metadata: {metadata_path}")

logger.info("\n🎉 Data warehouse is ready for analytics and visualization!")

2025-11-13 05:58:26,030 - INFO - 
2025-11-13 05:58:26,031 - INFO - ETL PIPELINE COMPLETED SUCCESSFULLY
2025-11-13 05:58:26,032 - INFO - ======================================================================
2025-11-13 05:58:26,036 - INFO - Total execution time: 780.41s
2025-11-13 05:58:26,037 - INFO - Database: /Users/shayanfareed/Developer/dwh/data_warehouse/global_health_datawarehouse.db
2025-11-13 05:58:26,037 - INFO - Exports: /Users/shayanfareed/Developer/dwh/data_warehouse/exports
2025-11-13 05:58:26,037 - INFO - Logs: /Users/shayanfareed/Developer/dwh/logs
2025-11-13 05:58:26,039 - INFO - Pipeline metadata: /Users/shayanfareed/Developer/dwh/logs/pipeline_metadata_20251113_055826.json
2025-11-13 05:58:26,039 - INFO - 
🎉 Data warehouse is ready for analytics and visualization!
